In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle


In [3]:
file_path = r"../data/LMA_Dataset.csv"

# Check if file exists
import os
if not os.path.exists(file_path):
	print(f"File not found: {file_path}")
	print("Available files in directory:")
	dir_path = os.path.dirname(file_path)
	if os.path.exists(dir_path):
		print(os.listdir(dir_path))
else:
	# Load the CSV (comment='%' skips COMSOL header lines)
	df = pd.read_csv(file_path, comment='%')

# Shows what have been loaded
print(f"Loaded {df.shape[0]:,} antenna designs!")
print(f"   Columns: {df.shape[1]}")
print()
print("First 3 rows:")
print(df.head(3))


Loaded 2,699 antenna designs!
   Columns: 22

First 3 rows:
   0.0037  0.0057  0.0002000000000000000  0.001574  3  0.029500000000000002  \
0  0.0037  0.0057                 0.0002  0.001574  3                0.0295   
1  0.0037  0.0057                 0.0002  0.001574  3                0.0295   
2  0.0037  0.0057                 0.0002  0.001574  3                0.0295   

   0.007  0.00003500000000000000  0.0005000000000000000  0.001  ...  \
0  0.007                0.000035                 0.0005  0.001  ...   
1  0.007                0.000035                 0.0005  0.001  ...   
2  0.007                0.000035                 0.0005  0.001  ...   

   2.076043617582667  0.008776300526959567  -5.849803356833777  \
0           2.072103              0.008782           -5.807565   
1           2.068062              0.008788           -5.765010   
2           2.063915              0.008794           -5.721852   

   5.996627711103428  7.779070873516634  -34.195759366625815  \
0        

In [4]:
# Load CSV 
df = pd.read_csv(file_path, skiprows=4)

# Fix the first column name (remove the %)
df.columns = df.columns.str.replace('% ', '')

print(f"Loaded {df.shape[0]:,} antenna designs!")
print(f"Columns: {df.shape[1]}")

# Define INPUT features (10 geometry parameters)
input_features = [
    'Lp (m)',
    'Wp (m)',
    'Wf (m)',
    'hs (m)',
    'er',
    'Ls (m)',
    'Ws (m)',
    'tp (m)',
    'tg (m)',
    'Lf (m)',
    'freq (GHz)'
]

# Define OUTPUT targets (8 performance metrics)
output_targets = [
    'S11(geometry) (1)',
    'Voltage standing wave ratio(geometry) (1)',
    'real part of zero impedence(geometry) (Ω)',
    'imaginarygeometry) (Ω)',
    'Maximum gain, dB(radiation) (1)',
    'Maximum directivity, dB(radiation) (1)',
    'Total radiated power, dB(radiation) (W)',
    '(radiation) (1)'  # Efficiency
]

# Extract as numpy arrays
X = df[input_features].values
Y = df[output_targets].values

print("\n Data separated!")
print(f"   X shape: {X.shape} → {X.shape[0]} designs × {X.shape[1]} inputs")
print(f"   Y shape: {Y.shape} → {Y.shape[0]} designs × {Y.shape[1]} outputs")


Loaded 2,700 antenna designs!
Columns: 22

 Data separated!
   X shape: (2700, 11) → 2700 designs × 11 inputs
   Y shape: (2700, 8) → 2700 designs × 8 outputs


In [5]:
# Count problems in the data
nan_in_X = np.isnan(X).sum()
nan_in_Y = np.isnan(Y).sum()
inf_in_X = np.isinf(X).sum()
inf_in_Y = np.isinf(Y).sum()
duplicates = df.duplicated().sum()

print("DATA QUALITY CHECK:")
print(f"   NaN in inputs (X):  {nan_in_X}")
print(f"   NaN in outputs (Y): {nan_in_Y}")
print(f"   Inf in inputs (X):  {inf_in_X}")
print(f"   Inf in outputs (Y): {inf_in_Y}")
print(f"   Duplicate rows:     {duplicates}")
print()

total_problems = nan_in_X + nan_in_Y + inf_in_X + inf_in_Y + duplicates

if total_problems == 0:
    print(" data is 100% clean!")
else:
    print(f"  Found {total_problems} problems")

print()



DATA QUALITY CHECK:
   NaN in inputs (X):  0
   NaN in outputs (Y): 0
   Inf in inputs (X):  0
   Inf in outputs (Y): 0
   Duplicate rows:     0

 data is 100% clean!



In [6]:
# Comprehensive data cleaning check
print("=" * 60)
print("COMPREHENSIVE DATA CLEANING CHECK")
print("=" * 60)
print()

# 1. Check for NaN values
print("1️ NaN VALUES:")
nan_count = df.isnull().sum()
if nan_count.sum() == 0:
    print("    No NaN values found")
else:
    print("    NaN values found:")
    print(nan_count[nan_count > 0])
print()

# 2. Check for infinite values
print("2️ INFINITE VALUES:")
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum()
if inf_count.sum() == 0:
    print("    No infinite values found")
else:
    print("    Infinite values found:")
    print(inf_count[inf_count > 0])
print()

# 3. Check for duplicates
print("3️ DUPLICATE ROWS:")
dup_count = df.duplicated().sum()
print(f"   {'yes' if dup_count == 0 else 'No'} Duplicate rows: {dup_count}")
print()

# 4. Check for outliers (using IQR method)
numeric_cols = df.select_dtypes(include=[np.number]).columns
outlier_count = 0
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    if outliers > 0:
        outlier_count += outliers
        print(f"    {col}: {outliers} outliers")

if outlier_count == 0:
    print("   No outliers detected")
print()

# 5. Data type check
print("5️ DATA TYPES:")
print(df.dtypes)
print()

# 6. Summary statistics
print("SUMMARY STATISTICS:")
print(df.describe())
print()

print("=" * 60)
print("Data appears CLEAN!" if (nan_count.sum() == 0 and inf_count.sum() == 0 and dup_count == 0) else "DATA QUALITY ISSUES FOUND")
print("=" * 60)

COMPREHENSIVE DATA CLEANING CHECK

1️ NaN VALUES:
    No NaN values found

2️ INFINITE VALUES:
    No infinite values found

3️ DUPLICATE ROWS:
   yes Duplicate rows: 0

    S11(geometry) (1): 91 outliers
    Maximum gain, dB(radiation) (1): 46 outliers
    Maximum directivity(radiation) (1): 13 outliers
    Maximum directivity, dB(radiation) (1): 5 outliers
    Total radiated power, dB(radiation) (W): 85 outliers
    (radiation) (1): 315 outliers

5️ DATA TYPES:
Lp (m)                                       float64
Wp (m)                                       float64
Wf (m)                                       float64
hs (m)                                       float64
er                                             int64
Ls (m)                                       float64
Ws (m)                                       float64
tp (m)                                       float64
tg (m)                                       float64
Lf (m)                                       float64
fr

In [7]:
# For Random Forest, use unscaled data
print("="*70)
print(" DATA SPLIT (for Random Forest)")
print("="*70)
print()

# Ensure X_clean and Y_clean are defined
if 'X_clean' not in locals():
	X_clean = X
if 'Y_clean' not in locals():
	Y_clean = Y

# Split the data: 70% train, 15% validation, 15% test
X_train, X_temp, Y_train, Y_temp = train_test_split(X_clean, Y_clean, test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

print("Data Split Complete (70-15-15)!")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape:   {X_val.shape}")
print(f"   X_test shape:  {X_test.shape}")
print()


 DATA SPLIT (for Random Forest)

Data Split Complete (70-15-15)!
   X_train shape: (1890, 11)
   X_val shape:   (405, 11)
   X_test shape:  (405, 11)



In [8]:
output_dir = 'RF_Trained_Data'
os.makedirs(output_dir, exist_ok=True)

np.save(f'{output_dir}/X_train.npy', X_train)
np.save(f'{output_dir}/X_val.npy', X_val)
np.save(f'{output_dir}/X_test.npy', X_test)
np.save(f'{output_dir}/Y_train.npy', Y_train)
np.save(f'{output_dir}/Y_val.npy', Y_val)
np.save(f'{output_dir}/Y_test.npy', Y_test)

print(f"Saved {output_dir}")

Saved RF_Trained_Data


In [9]:
# Load prepared data
X_train = np.load(os.path.join(output_dir, 'X_train.npy'))
X_val = np.load(os.path.join(output_dir, 'X_val.npy'))
X_test = np.load(os.path.join(output_dir, 'X_test.npy'))
Y_train = np.load(os.path.join(output_dir, 'Y_train.npy'))
Y_val = np.load(os.path.join(output_dir, 'Y_val.npy'))
Y_test = np.load(os.path.join(output_dir, 'Y_test.npy'))

print(f"Data loaded")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Data loaded
Train: (1890, 11), Val: (405, 11), Test: (405, 11)


#Train the random Forest Regressor #

In [10]:
import time


rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start = time.time()
rf_model.fit(X_train, Y_train)
train_time = time.time() - start

print(f" Trained in {train_time:.10f}s")

 Trained in 0.2390236855s


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished


In [11]:
Y_train_pred = rf_model.predict(X_train)
Y_test_pred = rf_model.predict(X_test)

train_r2 = r2_score(Y_train, Y_train_pred)
test_r2 = r2_score(Y_test, Y_test_pred)

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")

Train R²: 0.9950
Test R²: 0.9603


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


In [12]:
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
    
print(" Model saved")

 Model saved


In [ ]:
import time
import matplotlib.pyplot as plt

# Define output directory
output_dir = 'RF_Trained_Data'

# Load data
X_train = np.load(f'{output_dir}/X_train.npy')
X_val = np.load(f'{output_dir}/X_val.npy')
X_test = np.load(f'{output_dir}/X_test.npy')
Y_train = np.load(f'{output_dir}/Y_train.npy')
Y_val = np.load(f'{output_dir}/Y_val.npy')
Y_test = np.load(f'{output_dir}/Y_test.npy')

# Load model (using the same path as where it was saved in Cell 11)
model_path = 'rf_model.pkl'
with open(model_path, 'rb') as f:
    rf_model = pickle.load(f)

# Define feature names (OLD - without frequency)
input_features = ['Lp', 'Wp', 'Wf', 'hs', 'er', 'Ls', 'Ws', 'tp', 'tg', 'Lf']

print("="*70)
print("RANDOM FOREST MODEL ANALYSIS")
print("="*70)
print()

# 1. Model Performance
print("1️ MODEL PERFORMANCE:")
Y_train_pred = rf_model.predict(X_train)
Y_val_pred = rf_model.predict(X_val)
Y_test_pred = rf_model.predict(X_test)

train_r2 = r2_score(Y_train, Y_train_pred)
val_r2 = r2_score(Y_val, Y_val_pred)
test_r2 = r2_score(Y_test, Y_test_pred)

print(f"   Train R²: {train_r2:.4f}")
print(f"   Val R²:   {val_r2:.4f}")
print(f"   Test R²:  {test_r2:.4f}")
print()

# 2. Feature Importance
print("2️ FEATURE IMPORTANCE:")
importances = rf_model.feature_importances_
for name, imp in sorted(zip(input_features, importances), key=lambda x: x[1], reverse=True):
    print(f"   {name:10s}: {imp:.4f}")
print()

# 3. Per-output performance
output_names = ['S11', 'VSWR', 'Re(Z)', 'Im(Z)', 'Gain', 'Directivity', 'TRP', 'Efficiency']
print("3️ PER-OUTPUT R²:")
for i, name in enumerate(output_names):
    r2 = r2_score(Y_test[:, i], Y_test_pred[:, i])
    status = "yes" if r2 > 0.85 else "Warning" if r2 > 0.70 else "No"
    print(f"   {status} {name:15s}: {r2:.4f}")

print("="*70)

RANDOM FOREST MODEL ANALYSIS

1️⃣ MODEL PERFORMANCE:
   Train R²: 0.9950
   Val R²:   0.9494
   Test R²:  0.9603

2️⃣ FEATURE IMPORTANCE:
   hs        : 0.3338
   Lp        : 0.0776
   Wp        : 0.0041
   Wf        : 0.0016
   er        : 0.0000
   Ls        : 0.0000
   Ws        : 0.0000
   tp        : 0.0000
   tg        : 0.0000
   Lf        : 0.0000

3️⃣ PER-OUTPUT R²:
   yes S11            : 0.9831
   yes VSWR           : 0.9978
   yes Re(Z)          : 0.9973
   yes Im(Z)          : 0.9968
   yes Gain           : 0.9779
   Warning Directivity    : 0.7367
   yes TRP            : 0.9962
   yes Efficiency     : 0.9963


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


In [16]:
output_names = ['S11', 'VSWR', 'Re(Z)', 'Im(Z)', 'Gain', 'Directivity', 'TRP', 'Efficiency']

print("="*70)
print("PER-OUTPUT R² SCORES")
print("="*70)

for i, name in enumerate(output_names):
    train_r2_i = r2_score(Y_train[:, i], Y_train_pred[:, i])
    test_r2_i = r2_score(Y_test[:, i], Y_test_pred[:, i])
    
    status = "Pass" if test_r2_i > 0.85 else "Warning" if test_r2_i > 0.70 else "Fail"
    
    print(f"{status} {name:15s}: Train R²={train_r2_i:.4f}, Test R²={test_r2_i:.4f}")

print("="*70)

PER-OUTPUT R² SCORES
Pass S11            : Train R²=0.9986, Test R²=0.9831
Pass VSWR           : Train R²=0.9997, Test R²=0.9978
Pass Re(Z)          : Train R²=0.9996, Test R²=0.9973
Pass Im(Z)          : Train R²=0.9995, Test R²=0.9968
Pass Gain           : Train R²=0.9972, Test R²=0.9779
Warning Directivity    : Train R²=0.9668, Test R²=0.7367
Pass TRP            : Train R²=0.9995, Test R²=0.9962
Pass Efficiency     : Train R²=0.9995, Test R²=0.9963


In [17]:
print("="*70)
print("FEATURE VARIATION CHECK")
print("="*70)
print()

feature_names = ['Lp', 'Wp', 'Wf', 'hs', 'er', 'Ls', 'Ws', 'tp', 'tg', 'Lf']

print("Feature Statistics:")
print("="*70)
for i, name in enumerate(feature_names):
    min_val = X_train[:, i].min()
    max_val = X_train[:, i].max()
    std_val = X_train[:, i].std()
    unique_vals = len(np.unique(X_train[:, i]))
    
    # Check if feature varies
    varies = "VARIES" if std_val > 1e-10 else "CONSTANT"
    
    print(f"{name:10s}: min={min_val:.6f}, max={max_val:.6f}, std={std_val:.6f}, unique={unique_vals:4d} {varies}")

print("="*70)

FEATURE VARIATION CHECK

Feature Statistics:
Lp        : min=0.003700, max=0.003900, std=0.000081, unique=   3 VARIES
Wp        : min=0.005700, max=0.006000, std=0.000124, unique=   3 VARIES
Wf        : min=0.000200, max=0.000220, std=0.000010, unique=   2 VARIES
hs        : min=0.001574, max=0.001800, std=0.000113, unique=   2 VARIES
er        : min=3.000000, max=3.000000, std=0.000000, unique=   1 CONSTANT
Ls        : min=0.029500, max=0.029500, std=0.000000, unique=   1 CONSTANT
Ws        : min=0.007000, max=0.007000, std=0.000000, unique=   1 CONSTANT
tp        : min=0.000035, max=0.000035, std=0.000000, unique=   1 CONSTANT
tg        : min=0.000500, max=0.000500, std=0.000000, unique=   1 CONSTANT
Lf        : min=0.001000, max=0.001000, std=0.000000, unique=   1 CONSTANT
